# Point neuron circuit from EM

This notebook takes a set of **EM cell meshes**, resolves the connectivity between them directly from the EM dense reconstruction (via CAVE), writes a **Brian2 point-neuron SONATA circuit** in the same format as the Drosophila FlyWire example (`examples/J_drosophila`), and registers it as a `Circuit` entity.

The `PointNeuronCircuitFromEMTask`:
1. resolves each `EMCellMeshFromID` into its `pt_root_id`, source EM dataset and CAVE version,
2. checks that all meshes originate from the **same** EM dense reconstruction dataset,
3. queries the afferent synapses of each neuron and splits them into internal (between the modelled neurons) and external (from outside the set) connections,
4. writes a Brian2 SONATA circuit: a `brian2_point` population for the modelled neurons, a `virtual` population for the external presynaptic neurons, and `brian2_synapse` edge populations between them, and
5. registers it as a SONATA `Circuit` entity, taking the species, subject, brain region and brain region hierarchy from the EM dataset of the meshes.

The neuronal and synaptic parameters are borrowed from the Drosophila brain model as placeholders for now.

## Authentication

Authenticate against the OBI platform (`staging`) and select a project. The example EM cell meshes below are public on staging, so any project context with access works.

In [ ]:
import os
from pathlib import Path

import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token
from obi_notebook.get_projects import get_projects
from obi_one.config import settings
from obi_one.scientific.from_id.named_tuple_from_id import (
    PointNeuronCircuitFromEMInputNamedTuple,
)

environment = "staging"
token = get_token(environment=environment, auth_mode="daf")
project_context = ProjectContext(virtual_lab_id=obi.LAB_ID_STAGING_TEST, project_id=obi.PROJECT_ID_STAGING_TEST)

db_client = Client(
    token_manager=token, project_context=project_context, environment=environment
)

## CAVE token

Connectivity is queried from the EM dataset's CAVE datastack, which needs a CAVE token. If you do not already have one, you can request one with:

```python
import caveclient
caveclient.CAVEclient().auth.get_new_token()
```

Paste the token below. It is exposed to the task through the environment variable named by `settings.cave_client_config.microns_api_key` (`CAVECLIENT_MICRONS_API_KEY`).

In [ ]:
# from caveclient import CAVEclient
# client = CAVEclient()
# client.auth.setup_token(make_new=True)


os.environ[settings.cave_client_config.microns_api_key] = "1237d94cbabda5e5163478af43ad49bc"

## Select the EM cell meshes

We use four example EM cell meshes from staging. All four must originate from the same EM dense reconstruction dataset (the task enforces this).

In [ ]:
mesh_ids = [
    "ded69526-d15c-45f2-8aee-559095d76b94",
    "59088319-0106-4cd7-b739-64ecd8c64976",
    "dba8b32a-d369-4563-8f25-2e46590bf044",
    "9c7342cb-cb0d-40a7-8edc-2c2c54d27051",
]

cell_meshes = tuple(obi.EMCellMeshFromID(id_str=mesh_id) for mesh_id in mesh_ids)

## Alternative: all EM cell meshes for the IARPA dataset (release 1718)

Instead of the hand-picked IDs above, query **all** EM cell meshes registered for the IARPA MICrONS dataset at release version 1718 (the `release_version` property of the `EMCellMesh` entity) and use them as the input (this overrides `cell_meshes`). Each modelled neuron triggers a CAVE synapse query, so a larger set takes proportionally longer.

In [ ]:
from entitysdk import models

# All EM dense reconstruction datasets visible in this project, then pick the IARPA one.
datasets = db_client.search_entity(entity_type=models.EMDenseReconstructionDataset).all()
iarpa_datasets = [d for d in datasets if "IARPA" in (d.name or "")]
print("IARPA datasets:", [d.name for d in iarpa_datasets])
iarpa_dataset = iarpa_datasets[0]
print("Using EM dataset:", iarpa_dataset.name)

# Query all EM cell meshes for that dataset at release version 1718 (EMCellMesh.release_version).
release_version = 1718
iarpa_meshes = db_client.search_entity(
    entity_type=models.EMCellMesh,
    query={
        "em_dense_reconstruction_dataset__name__ilike": iarpa_dataset.name,
        "release_version": release_version,
    },
).all()
print(
    f"Found {len(iarpa_meshes)} EM cell meshes for {iarpa_dataset.name!r} "
    f"at release version {release_version}."
)

# Keep just one mesh per dense_reconstruction_cell_id (the task rejects duplicate cell ids).
unique_meshes = {}
for mesh in iarpa_meshes:
    unique_meshes.setdefault(mesh.dense_reconstruction_cell_id, mesh)
print(f"Using {len(unique_meshes)} meshes after de-duplicating by dense_reconstruction_cell_id.")

import random

# Subsample cell_meshes to a fraction of neurons for quick testing (set to 1.0 to use all).
sample_fraction = 1.0
if sample_fraction < 1.0:
    rng = random.Random(0)
    n_sample = max(1, round(sample_fraction * len(unique_meshes)))
    unique_meshes = dict(rng.sample(list(unique_meshes.items()), n_sample))

print(f"Using {len(unique_meshes)} cell meshes (sample_fraction={sample_fraction}).")

cell_meshes = tuple(obi.EMCellMeshFromID(id_str=str(mesh.id)) for mesh in unique_meshes.values())

## Build the config and run the task

All meshes are grouped into a single named tuple so that they form **one** circuit (rather than being expanded into separate scan coordinates). The Brian2 SONATA circuit is written under `coordinate_output_root`.

In [ ]:
# Output directory for the generated Brian2 SONATA circuit (adjust as you like).
output_dir = Path("point_neuron_circuit_from_em_output")

config = obi.PointNeuronCircuitFromEMSingleConfig(
    info=obi.Info(
        campaign_name="Point neuron circuit from EM",
        campaign_description="Resolve connectivity between a set of EM cell meshes.",
    ),
    initialize=obi.PointNeuronCircuitFromEMSingleConfig.Initialize(
        cell_meshes=PointNeuronCircuitFromEMInputNamedTuple(
            name="EM meshes", elements=cell_meshes
        ),
    ),
    coordinate_output_root=output_dir,
)

task = obi.PointNeuronCircuitFromEMTask(config=config)
task.execute(db_client=db_client)

## Inspect the resolved connectivity

The resolved connectivity is also available on the task as pandas DataFrames for further analysis.

In [ ]:
task.internal_connectivity

In [ ]:
task.neuron_summary

## Inspect the generated Brian2 SONATA circuit

List the written files and load the circuit with bluepysnap to confirm the point-neuron and virtual populations and the synapse edge populations.

In [ ]:
print("circuit_config:", task.circuit_config_path)
print("registered circuit id:", task.registered_circuit_id)
for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(output_dir))

In [ ]:
import bluepysnap

circuit = bluepysnap.Circuit(str(task.circuit_config_path))
for pop in circuit.nodes.population_names:
    nodes = circuit.nodes[pop]
    print(f"node population {pop!r}: {nodes.size} nodes (type={nodes.type})")
for pop in circuit.edges.population_names:
    print(f"edge population {pop!r}: {circuit.edges[pop].size} edges")